# Notebook 03 — Weather Fetch

**Goal:** For each unique (city, date) pair in the marathon dataset, fetch historical weather data from the Open-Meteo API and produce a `weather.csv` file.

**API:** https://open-meteo.com — free, no key required.

**Variables fetched per day:**
- `temperature_2m_mean` — mean air temperature (°C)
- `temperature_2m_max` — max temperature (°C, proxy for race-time heat)
- `precipitation_sum` — total precipitation (mm)
- `windspeed_10m_max` — max wind speed (km/h)
- `relative_humidity_2m_mean` — mean relative humidity (%)

City coordinates are resolved via the Open-Meteo geocoding API.

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import os
import json

PROC_DIR = '../data/processed'
os.makedirs(PROC_DIR, exist_ok=True)

MARATHON_CLEAN = os.path.join(PROC_DIR, 'marathon_clean.csv')
WEATHER_OUT    = os.path.join(PROC_DIR, 'weather.csv')
WEATHER_CACHE  = os.path.join(PROC_DIR, 'weather_cache.json')  # avoid re-fetching

## 1. Load Marathon Data & Identify (city, date) Pairs

In [2]:
marathon = pd.read_csv(MARATHON_CLEAN, low_memory=False)
print(f'Marathon rows: {len(marathon):,}')

city_col = 'city_clean' if 'city_clean' in marathon.columns else 'city'

# Unique (city, date) pairs — one API call per pair
pairs = (
    marathon[[city_col, 'race_date']]
    .dropna(subset=[city_col, 'race_date'])
    .drop_duplicates()
    .rename(columns={city_col: 'city'})
    .reset_index(drop=True)
)
print(f'Unique (city, date) pairs to fetch: {len(pairs)}')
print(pairs)

Marathon rows: 2,854,273


Unique (city, date) pairs to fetch: 89
           city   race_date
0   Minneapolis  2000-10-01
1   Minneapolis  2001-10-07
2   Minneapolis  2002-10-06
3   Minneapolis  2006-10-01
4   Minneapolis  2007-10-07
..          ...         ...
84      Chicago  2012-10-07
85      Chicago  2013-10-13
86      Chicago  2014-10-12
87      Chicago  2016-10-09
88      Chicago  2017-10-08

[89 rows x 2 columns]


## 2. City → Coordinates via Open-Meteo Geocoding

In [3]:
# Known coordinates for the US marathon cities (avoids extra API calls)
CITY_COORDS = {
    'Boston':          (42.3601, -71.0589),
    'Chicago':         (41.8781, -87.6298),
    'New York City':   (40.7128, -74.0060),
    'Los Angeles':     (34.0522, -118.2437),
    'Washington':      (38.9072, -77.0369),
    'Minneapolis':     (44.9778, -93.2650),
    'Duluth':          (46.7867, -92.1005),
    'Houston':         (29.7604, -95.3698),
    'Monterey':        (36.6002, -121.8947),
}

def get_coords(city):
    if city in CITY_COORDS:
        return CITY_COORDS[city]
    # Fallback: Open-Meteo geocoding
    url = f'https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&language=en&format=json'
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    results = r.json().get('results', [])
    if results:
        return results[0]['latitude'], results[0]['longitude']
    raise ValueError(f'City not found: {city}')

for city in pairs['city'].unique():
    lat, lon = get_coords(city)
    print(f'{city:20s} → ({lat:.4f}, {lon:.4f})')

Minneapolis          → (44.9778, -93.2650)
Duluth               → (46.7867, -92.1005)
Houston              → (29.7604, -95.3698)
New York City        → (40.7128, -74.0060)
Chicago              → (41.8781, -87.6298)


## 3. Fetch Historical Weather from Open-Meteo

In [4]:
def fetch_weather(city, date_str):
    """Fetch daily weather for a single city on a single date."""
    lat, lon = get_coords(city)
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'start_date': date_str,
        'end_date':   date_str,
        'daily': ','.join([
            'temperature_2m_mean',
            'temperature_2m_max',
            'precipitation_sum',
            'windspeed_10m_max',
            'relative_humidity_2m_mean',
        ]),
        'timezone': 'America/New_York',
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    data = r.json().get('daily', {})
    if not data or not data.get('time'):
        return None
    return {
        'city':         city,
        'race_date':    date_str,
        'temp_mean_c':  data['temperature_2m_mean'][0],
        'temp_max_c':   data['temperature_2m_max'][0],
        'precip_mm':    data['precipitation_sum'][0],
        'wind_kmh':     data['windspeed_10m_max'][0],
        'humidity_pct': data.get('relative_humidity_2m_mean', [None])[0],
    }

In [5]:
# Load cache if it exists (lets you resume after interruption)
cache = {}
if os.path.exists(WEATHER_CACHE):
    with open(WEATHER_CACHE) as f:
        cache = json.load(f)
    print(f'Loaded {len(cache)} cached weather records')

records = []
for _, row in pairs.iterrows():
    key = f"{row['city']}|{row['race_date']}"
    if key in cache:
        records.append(cache[key])
        continue
    try:
        rec = fetch_weather(row['city'], row['race_date'])
        if rec:
            cache[key] = rec
            records.append(rec)
            print(f"  Fetched {row['city']} {row['race_date']} — temp={rec['temp_mean_c']}°C")
        else:
            print(f"  No data for {row['city']} {row['race_date']}")
    except Exception as e:
        print(f"  ERROR {row['city']} {row['race_date']}: {e}")
    time.sleep(0.3)  # be polite to the API

# Save cache
with open(WEATHER_CACHE, 'w') as f:
    json.dump(cache, f, indent=2)

weather_df = pd.DataFrame(records)
print(f'\nWeather records: {len(weather_df)}')
weather_df.head()

Loaded 173 cached weather records


  Fetched Houston 2002-01-20 — temp=9.3°C



Weather records: 89


,city,race_date,temp_mean_c,temp_max_c,precip_mm,wind_kmh,humidity_pct
0,Minneapolis,2000-10-01,19.8,23.6,1.1,17.2,NaN
1,Minneapolis,2001-10-07,5.9,12.5,0.0,18.8,NaN
2,Minneapolis,2002-10-06,7.6,8.9,17.3,28.4,NaN
3,Minneapolis,2006-10-01,16.1,24.8,0.0,21.0,NaN
4,Minneapolis,2007-10-07,24.0,27.0,1.6,20.5,NaN


## 4. Quick Sanity Check

In [6]:
print('Temperature range (°C):')
print(weather_df['temp_mean_c'].describe())
print('\nMissing values:')
print(weather_df.isnull().sum())

Temperature range (°C):
count    89.000000
mean     12.778652
std       5.183770
min       3.700000
25%       8.400000
50%      12.400000
75%      16.100000
max      25.100000
Name: temp_mean_c, dtype: float64

Missing values:
city             0
race_date        0
temp_mean_c      0
temp_max_c       0
precip_mm        0
wind_kmh         0
humidity_pct    87
dtype: int64


## 5. Save

In [7]:
weather_df.to_csv(WEATHER_OUT, index=False)
print(f'Saved {WEATHER_OUT} — {len(weather_df)} rows')

Saved ../data/processed/weather.csv — 89 rows
